# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [3]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

# Use the cleaned CSV file
loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains_cleaned.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [39]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [ ]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [13]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [14]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [15]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [16]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [17]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the project domains include Security, Healthcare / MedTech, Productivity Assistants, Creative / Design / Media, E‑commerce / Marketplaces, Developer Tools / DevEx, Writing & Content, and Customer Support / Helpdesk. \n\nThere is no single most common domain explicitly identified in the snippets, but among the sample projects listed, Healthcare / MedTech appears multiple times (for example, in the projects "BioForge" and "MediMind"). \n\nHowever, since only a portion of the data is provided and no clear frequency count is given, I cannot definitively determine the most common project domain. \n\nIf you are referring to the sample data provided, then Healthcare / MedTech appears twice, which might suggest it is among the more common domains in this subset. \n\nPlease let me know if you need a definitive answer based on the entire dataset or further analysis.'

In [18]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are several use cases related to security. Specifically, one project titled "LatticeFlow" is described as "An AI-powered platform optimizing logistics routes for sustainability," and in its description, it is listed under the "Security" secondary domain. Therefore, yes, there are use cases about security.'

In [19]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had various positive comments about the fintech projects. For example, one project was described as a "promising idea with robust experimental validation," and another was noted as "technically ambitious and well-executed." Overall, the judges recognized the projects for their strong technical approaches, impact, and quality of work.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [21]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [27]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain is not explicitly stated, but from the sample, the Domains mentioned are Productivity Assistants, Legal / Compliance, Data / Analytics, and Healthcare / MedTech. Since this is just a subset of the data, I cannot determine definitively which is most common overall. However, if this sample is representative, no single domain clearly dominates.\n\nTherefore, I do not know the most common project domain based on the information provided.'

In [28]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any specific use cases related to security.'

In [29]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges described the fintech-related project "PulseAI 50" as "technically ambitious and well-executed."'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer
BM25 is good in situations where we need classical search capabilities like scoring a document based on how many times a keyword occurred in a document. Its useful in situations where exact keyword match is required. For example question - `Which projects used FastAPI?`. In this case we want only projects that have fastAPI and not necessarily project that might be alternatives to fastapi like django or flask.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [25]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [26]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [30]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, there is no indication of a single most common project domain. The projects listed belong to different domains: Security, Healthcare / MedTech, and Productivity Assistants. Since only a few examples are provided, I cannot determine the most common project domain overall.'

In [31]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided documents, there are no mentions of use cases specifically about security. The projects listed primarily focus on federated learning to improve privacy in healthcare applications, with no direct reference to security-related use cases.'

In [32]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech project "Pathfinder 27." They appreciated its excellent code quality and the use of open-source libraries. The project received a high score of 81 and a judge score of 9.8, indicating strong approval.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [34]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [35]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," which is mentioned multiple times throughout the documents.'

In [36]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are usecases related to security. Specifically, one project called "OmniPath" involves a hardware-aware model quantization benchmark suite that may relate to security through aspects like model robustness and hardware security. Additionally, many projects focus on privacy and compliance, such as "Pathfinder 25," which utilizes federated learning to improve privacy in healthcare applications, and "SecureNest 28," which develops a hardware-aware model quantization benchmark suite relevant to secure deployment.'

In [37]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had mixed but generally positive comments about the fintech projects. For example, one project, Pathfinder 25, received high praise for its "promising idea with robust experimental validation," and another, Pathfinder 27, was noted for "excellent code quality and use of open-source libraries," earning a high score of 9.8. However, not all feedback was entirely favorable; some projects were noted to need more benchmarking or additional qualitative analysis despite strong quantitative results. Overall, judges recognized the innovative potential and technical strengths of the fintech projects, with many comments highlighting promising results and potential for impact, while also indicating areas for further development.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer
1) Multiple reformulations can help take into account different ways of describing the same concept. By using multiple reformulations that have synonyms and semantically related terms, the system can retrieve documents that do not contain the original query but may still be relevant to the user's intent. 
2) Users may not always use the same terms and so having multiple formulations of the same query can help match documents that a single query cannot.
3) A single short query can be interpreted in multiple ways. Generating a response for each reformulation can help the system better respond to the search intent.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [40]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [41]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [42]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [43]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [44]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [45]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned more than once among the example projects.'

In [46]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any specific usecases explicitly related to security. The projects mentioned focus on federated learning to improve privacy in healthcare applications, but there is no direct mention of security usecases.'

In [47]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Based on the provided context, the judges had the following comments about the fintech projects:\n\n- For the project "SkyForge" in the finance/fintech domain, the judges described it as "A clever solution with measurable environmental benefit."\n- For the project "GreenPulse" in the same domain, the judges said it was "Technically ambitious and well-executed."\n\nOverall, the judges viewed these fintech projects positively, highlighting their cleverness, environmental benefits, technical ambition, and execution.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [48]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [49]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [50]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times in the examples. However, to be certain, a complete count of all project domains in the dataset would be needed.  \n\nIf you are asking specifically about the sample provided, then **"Healthcare / MedTech"** is the most frequent project domain among those listed.'

In [51]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there was a use case related to security. Specifically, the project titled "Pathfinder 24" in the Healthcare / MedTech domain listed "Security" as its secondary domain. Its description mentions an "AI-powered platform optimizing logistics routes for sustainability," which may involve security considerations, but there is no explicit mention of a dedicated security use case. \n\nAdditionally, another project titled "SecureNest 49" in the E‑commerce / Marketplaces domain, with "Legal / Compliance" as a secondary domain, could imply security and compliance aspects related to enterprise knowledge bases, but again, there is no explicit focus solely on security.\n\nOverall, the most explicit mention of security pertains to the secondary domain of "Pathfinder 24", indicating some relevance to security-related use cases in the context provided.'

In [52]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges\' comments on the fintech projects were generally positive. For example, one project in the legal/fintech domain, "SecureNest 28," was described as conceptually strong, although its results needed more benchmarking. Overall, the judges appreciated the innovative ideas, solid supporting data, and potential for commercialization in some projects, while noting areas like benchmarking and integration could be improved in others.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [53]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [54]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [55]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [56]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [57]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [58]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Legal / Compliance," which is mentioned twice. Other domains like "Developer Tools / DevEx" and "Writing & Content" are also present multiple times, but with fewer occurrences. Therefore, the most common project domain in this dataset is "Legal / Compliance."'

In [59]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, the project titled "BioForge" falls under the Security domain and involves a medical imaging solution that improves early diagnosis through vision transformers. Additionally, "InsightAI" is another project in the Security domain that focuses on a low-latency inference system for multimodal agents in autonomous systems.'

In [60]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had positive comments about the fintech projects, highlighting their technical maturity and potential. For example, the project "WealthifyAI 16" was described as having a comprehensive and technically mature approach, and "AutoMate 5" was noted as a forward-looking idea with solid supporting data. Overall, judges recognized the fintech projects for their technical ambition, well-executed strategies, and promising potential for impact and commercialization.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer
if sentences are highly repetitive, we may end up with all similar senetences in one chunk leading to  very large sized chunks which could lead to large context windows and more token usage (costly). We could add a max chuk size to limit the size of the chunks.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [53]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

## Synthetic Data Generation with RAGAS
We will first start by creating the synthetic data. I was running into the 100 token limit for ragas and class mates suggested that I augment that description with the metadata in the csv file i.e use things like title, domain etc to create a well structured description. So that is what I am doing below.

In [74]:
### YOUR CODE HERE
import pandas as pd
from langchain_core.documents import Document

# Load CSV data using pandas for cleaner parsing
df = pd.read_csv("data/Projects_with_Domains_cleaned.csv")

# Strategy: Combine multiple projects into larger documents to reduce duplication issues
# This will create fewer documents with more diverse content
expanded_data_set = []
chunk_size = 3  # Combine 3 projects per document

for chunk_idx in range(0, len(df), chunk_size):
    chunk_df = df.iloc[chunk_idx:chunk_idx + chunk_size]
    
    # Build a combined narrative for multiple projects
    project_narratives = []
    
    for idx, row in chunk_df.iterrows():
        # Extract data from DataFrame row (convert to strings to avoid tuples)
        project_title = str(row['Project Title']) if pd.notna(row['Project Title']) else 'Unknown Project'
        project_name = str(row['Project Name']) if pd.notna(row['Project Name']) else 'Unknown'
        domain = str(row['Project Domain']) if pd.notna(row['Project Domain']) else 'General'
        secondary_domain = str(row['Secondary Domain']) if pd.notna(row['Secondary Domain']) else 'N/A'
        description = str(row['Description']) if pd.notna(row['Description']) else 'No description available.'
        judge_comments = str(row['Judge Comments']) if pd.notna(row['Judge Comments']) else 'No comments provided.'
        score = str(row['Score']) if pd.notna(row['Score']) else '0'
        judge_score = str(row['Judge Score']) if pd.notna(row['Judge Score']) else '0.0'
        
        # Keep original domain names with slashes - just normalize special dash characters
        domain_clean = domain.replace('‑', '-')
        secondary_domain_clean = secondary_domain.replace('‑', '-')
        
        # Create a narrative story for this project
        narrative = f"""Project: {project_title}
{project_title} (codename '{project_name}') operates in {domain_clean} with applications in {secondary_domain_clean}. {description} The project scored {score}/100 (judge rating: {judge_score}/10). Judge feedback: {judge_comments}"""
        
        project_narratives.append(narrative)
    
    # Combine all project narratives in this chunk into one document
    combined_narrative = "\n\n---\n\n".join(project_narratives)
    
    # Create document with combined projects
    expanded_doc = Document(
        page_content=combined_narrative,
        metadata={
            "source": "projects",
            "doc_id": str(chunk_idx // chunk_size),
            "num_projects": str(len(chunk_df))
        }
    )
    expanded_data_set.append(expanded_doc)

# Also create the docs variable for compatibility with downstream code
docs = expanded_data_set

print(f"Created {len(expanded_data_set)} expanded narrative documents from {len(df)} CSV rows")
print(f"Average document length: {sum(len(doc.page_content) for doc in expanded_data_set) // len(expanded_data_set)} characters")
print(f"\nSample expanded content:\n{'-'*80}\n{expanded_data_set[0].page_content[:600]}...")

Created 17 expanded narrative documents from 50 CSV rows
Average document length: 973 characters

Sample expanded content:
--------------------------------------------------------------------------------
Project: InsightAI 1
InsightAI 1 (codename 'Project Aurora') operates in Security with applications in Finance & FinTech. A low-latency inference system for multimodal agents in autonomous systems. The project scored 85/100 (judge rating: 9.5/10). Judge feedback: Technically ambitious and well-executed.

---

Project: ShopSmart 2
ShopSmart 2 (codename 'OmniPath') operates in Developer Tools & DevEx with applications in Productivity Assistants. A simulation environment for embodied AI agents using Unreal integration. The project scored 67/100 (judge rating: 8.1/10). Judge feedback: Excellent te...


In [75]:
expanded_data_set[0].metadata

{'source': 'projects', 'doc_id': '0', 'num_projects': '3'}

##Personas 
First we will create personas for our data set. I created 3 personas - 
1) 


In [76]:
## personas go here and then regener
from ragas.testset.persona import Persona

persona_novice= Persona(
    name="Novice builder",
    role_description="Don't know much about AI and is looking for information on how to get started on an AI project ",
)
persona_seasoned = Persona(
    name="Seasoned builder",
    role_description="Knows about AI and has built a few projects but is looking for more information on how to improve their skills",
)
persona_expert = Persona(
    name="Expert builder",
    role_description="Seasoned AI builder who has built many projects and is looking for more projects that they can build.",
)

personas = [persona_novice, persona_seasoned, persona_expert]
personas

[Persona(name='Novice builder', role_description="Don't know much about AI and is looking for information on how to get started on an AI project "),
 Persona(name='Seasoned builder', role_description='Knows about AI and has built a few projects but is looking for more information on how to improve their skills'),
 Persona(name='Expert builder', role_description='Seasoned AI builder who has built many projects and is looking for more projects that they can build.')]

In [77]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/var/folders/dq/7_w0fl7j3s3fs7t6pbr2jtmh0000gq/T/ipykernel_38411/2347249996.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
/var/folders/dq/7_w0fl7j3s3fs7t6pbr2jtmh0000gq/T/ipykernel_38411/2347249996.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


In [78]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(expanded_data_set, testset_size=10)


Applying SummaryExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/17 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

ValidationError: 2 validation errors for ThemesPersonasInput
themes.0
  Input should be a valid string [type=string_type, input_value=('WealthifyAI 40', 'WealthifyAI 46'), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
themes.1
  Input should be a valid string [type=string_type, input_value=('SynthMind', 'SynthMind'), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [ ]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How are project evaluation and project domains...,[<1-hop>\n\nProject 1:\nProject Title: \nDomai...,"For each project submission, project evaluatio...",multi_hop_abstract_query_synthesizer
1,wher is projct evalution and domans writen?,[<1-hop>\n\nProject 1:\nProject Title: \nDomai...,Projct evalution and domans are writen in the ...,multi_hop_abstract_query_synthesizer
2,How are project evaluation and project domains...,[<1-hop>\n\nProject 1:\nProject Title: \nDomai...,Project evaluation and project domains are rec...,multi_hop_abstract_query_synthesizer
